# Pipeline 100% Classique — v3 : Segmentation Spatiale

## Problème fondamental de v1/v2

```
v1/v2 : image = sac de pixels global
        HOG(entier) + HSV(entier) + LBP(entier)
        → on sait QUE l'image contient du rouge
        → on ne sait PAS OÙ est le rouge
```

Or la cohérence texte-image dépend de la **position des objets** :
- *"A dog runs on the left"* → cohérent si objet en bas-gauche, incohérent si en haut-droite
- *"A sunset sky"* → cohérent si couleurs chaudes EN HAUT, incohérent si en bas

## Solutions v3

### 1. Grille 3×3 (9 zones)
Chaque feature image est calculée **localement** dans chacune des 9 zones :
- HOG local, HSV local, LBP local → × 9
- Densité de contours Canny par zone
- Énergie de texture par zone

### 2. Segmentation multi-méthodes
| Méthode | Ce qu'elle capture |
|---|---|
| **Canny** | Contours nets, bords francs |
| **Deriche** (approx via Scharr) | Contours lisses, gradient orienté |
| **Seuillage adaptatif** | Régions sombre/claire localement |
| **Otsu** | Séparation fond/objet global |
| **Watershed** (simplifié) | Bassins versants = objets distincts |
| **Saliency map** (gradient) | Zones qui attirent l'attention |

### 3. Features de localisation d'objets
- **Centre de masse** des contours (position dominante des objets)
- **Distribution spatiale** : entropie de la carte de saillance
- **Asymétrie gauche/droite et haut/bas** (composition)
- **Nombre de composantes connexes** par zone (= combien d'objets distincts)
- **Moments spatiaux** (μ_x, μ_y, σ_x, σ_y des contours)

### 4. Features texte enrichies
- **Mots de position** : left, right, top, bottom, center, behind, front...
- **Mots de couleur** : red, blue, sky, dark, bright...
- **Mots d'objet concret** (noms propres spatiaux)

### 5. Cross-features spatiaux texte↔image
- Si le texte mentionne "left" → comparer avec l'activité en zone gauche
- Si le texte mentionne "sky/top" → comparer avec les couleurs en zone haute

**Objectif : 65% → 75%+ en restant 100% classique**

In [1]:
!pip install opencv-python scikit-image vaderSentiment --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 — Imports
# ═══════════════════════════════════════════════════════════════════
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from skimage.color import rgb2gray
from skimage import io, transform, filters, morphology, measure
from skimage.filters import threshold_otsu, threshold_local, scharr
from skimage.measure import label, regionprops
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
from scipy.stats import entropy as scipy_entropy, spearmanr, pearsonr
from scipy.spatial.distance import cosine, cityblock, chebyshev, correlation, braycurtis, canberra

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cross_decomposition import CCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_OK = True
    vader = SentimentIntensityAnalyzer()
except ImportError:
    VADER_OK = False

warnings.filterwarnings('ignore')
for r in ['punkt','stopwords','wordnet','omw-1.4',
          'averaged_perceptron_tagger_eng','punkt_tab']:
    nltk.download(r, quiet=True)

lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words('english'))

print("Imports OK — Pipeline v3 Spatial")

[nltk_data] Error loading punkt: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>
[nltk_data] Error loading stopwords: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>
[nltk_data] Error loading wordnet: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>
[nltk_data] Error loading omw-1.4: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>
[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [Errno -2] Name or service not known>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>


LookupError: 
**********************************************************************
  Resource [93mstopwords[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('stopwords')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/stopwords[0m

  Searched in:
    - '/usr/local/share/nltk_data'
    - '/root/nltk_data'
    - '/usr/local/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/local/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — Chargement des données
# ═══════════════════════════════════════════════════════════════════
DATA_DIR = '../data/processed'

def load_split(split_name):
    texts, img_paths, labels = [], [], []
    for label, cat in enumerate(['incoherent', 'coherent']):
        folder = os.path.join(DATA_DIR, split_name, cat)
        if not os.path.exists(folder):
            print(f"Dossier {folder} non trouvé. Skipping.")
            continue
        for f in sorted(os.listdir(folder)):
            if f.endswith('.txt'):
                with open(os.path.join(folder, f), 'r', encoding='utf-8') as fh:
                    texts.append(fh.read().strip())
                img_paths.append(os.path.join(folder, f.replace('.txt', '.jpg')))
                labels.append(label)
    texts, img_paths, labels = shuffle(texts, img_paths,
                                       np.array(labels), random_state=42)
    return np.array(texts), np.array(img_paths), labels

print("Chargement des splits...")
t_train, p_train, y_train = load_split('train')
t_val,   p_val,   y_val   = load_split('validation')
t_test,  p_test,  y_test  = load_split('test')

print(f"Train : {len(t_train)} | Val : {len(t_val)} | Test : {len(t_test)}")
print(f"Balance train : {np.bincount(y_train)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Fonctions de segmentation et grille spatiale
# ═══════════════════════════════════════════════════════════════════

IMG_SIZE   = (192, 192)   # plus grand pour la grille 3×3 (64×64 par zone)
GRID_N     = 3            # grille 3×3 = 9 zones
HOG_PIXELS = 8            # plus fin pour les zones locales
LBP_RADIUS = 2
LBP_POINTS = 8 * LBP_RADIUS
N_HSV_BINS = 16           # moins de bins par zone (zone = 64×64)


# ── Helpers grille ────────────────────────────────────────────────
def get_grid_zones(img, n=GRID_N):
    """Retourne les n² sous-images de la grille n×n."""
    h, w = img.shape[:2]
    zh, zw = h // n, w // n
    zones = []
    for r in range(n):
        for c in range(n):
            zone = img[r*zh:(r+1)*zh, c*zw:(c+1)*zw]
            zones.append((r, c, zone))
    return zones  # liste de (row, col, patch)


# ── Segmentation multi-méthodes ───────────────────────────────────
def segment_canny(gray, low=50, high=150):
    """Canny : contours nets. Retourne carte binaire."""
    g8 = (gray * 255).astype(np.uint8)
    return cv2.Canny(g8, low, high).astype(float) / 255.0


def segment_deriche(gray):
    """
    Approximation Deriche via filtre Scharr (même philosophie :
    gradient lissé optimisé). Retourne magnitude normalisée.
    """
    mag = scharr(gray)
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag


def segment_adaptive(gray):
    """Seuillage adaptatif local (block_size=35)."""
    g8 = (gray * 255).astype(np.uint8)
    block = min(35, (min(gray.shape[:2]) // 4) | 1)  # doit être impair
    thresh = threshold_local(g8, block_size=block, offset=10)
    return (g8 > thresh).astype(float)


def segment_otsu(gray):
    """Seuillage Otsu global : sépare fond/objet."""
    thresh = threshold_otsu(gray)
    return (gray > thresh).astype(float)


def segment_watershed(gray):
    """
    Watershed simplifié :
    1. Otsu → masque binaire
    2. Distance transform
    3. Marqueurs = maxima locaux
    4. Watershed
    Retourne le nombre de régions et leur carte.
    """
    binary = segment_otsu(gray).astype(bool)
    distance = ndi.distance_transform_edt(binary)
    # peak_local_max avec min_distance adaptatif
    min_d = max(5, min(gray.shape) // 10)
    coords = peak_local_max(distance, min_distance=min_d, labels=binary)
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)
    labels_ws = watershed(-distance, markers, mask=binary)
    return labels_ws, markers


def compute_saliency_map(gray):
    """
    Saliency map par gradient spectral (approx simple) :
    - FFT → amplitude → log → IFFT → carte d'attention.
    Zones saillantes = où l'image est "surprenante".
    """
    g8 = (gray * 255).astype(np.float32)
    fft  = np.fft.fft2(g8)
    log_amp = np.log(np.abs(fft) + 1e-8)
    # Résidu spectral
    kernel = np.ones((3,3), np.float32) / 9
    smooth = cv2.filter2D(log_amp, -1, kernel)
    spectral_residual = log_amp - smooth
    # Reconstruire avec phase originale
    reconstructed = np.fft.ifft2(
        np.exp(spectral_residual + 1j * np.angle(fft))
    ).real
    saliency = cv2.GaussianBlur(reconstructed**2, (9,9), 2.5)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency


print("Fonctions de segmentation définies.")

# ── Visualisation sur un exemple ──────────────────────────────────
def visualize_segmentation(img_path):
    img  = io.imread(img_path)
    if img.ndim == 2: img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4: img = img[:,:,:3]
    img  = (transform.resize(img, IMG_SIZE, anti_aliasing=True) * 255).astype(np.uint8)
    gray = rgb2gray(img)

    fig, axes = plt.subplots(2, 5, figsize=(22, 9))

    axes[0,0].imshow(img);                         axes[0,0].set_title('Original', fontweight='bold')
    axes[0,1].imshow(segment_canny(gray),    cmap='gray'); axes[0,1].set_title('Canny')
    axes[0,2].imshow(segment_deriche(gray),  cmap='hot');  axes[0,2].set_title('Deriche (Scharr)')
    axes[0,3].imshow(segment_adaptive(gray), cmap='gray'); axes[0,3].set_title('Seuillage Adaptatif')
    axes[0,4].imshow(segment_otsu(gray),     cmap='gray'); axes[0,4].set_title('Otsu')

    ws_map, markers = segment_watershed(gray)
    sal = compute_saliency_map(gray)
    axes[1,0].imshow(ws_map, cmap='nipy_spectral'); axes[1,0].set_title('Watershed')
    axes[1,1].imshow(sal, cmap='hot');              axes[1,1].set_title('Saliency Map')

    # Grille 3×3 sur l'image originale
    axes[1,2].imshow(img)
    h, w = img.shape[:2]
    zh, zw = h//GRID_N, w//GRID_N
    for r in range(GRID_N):
        for c in range(GRID_N):
            rect = patches.Rectangle((c*zw, r*zh), zw, zh,
                                      linewidth=2, edgecolor='yellow', facecolor='none')
            axes[1,2].add_patch(rect)
            axes[1,2].text(c*zw+zw//2, r*zh+zh//2, f'Z{r*3+c}',
                           ha='center', va='center', color='yellow', fontsize=12, fontweight='bold')
    axes[1,2].set_title('Grille 3×3 (9 zones)', fontweight='bold')

    # Heatmap densité de contours Canny par zone
    canny = segment_canny(gray)
    grid_canny = np.array([
        [canny[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
         for c in range(GRID_N)] for r in range(GRID_N)
    ])
    axes[1,3].imshow(grid_canny, cmap='YlOrRd', interpolation='nearest')
    for r in range(GRID_N):
        for c in range(GRID_N):
            axes[1,3].text(c, r, f'{grid_canny[r,c]:.2f}',
                           ha='center', va='center', fontsize=11, fontweight='bold')
    axes[1,3].set_title('Densité contours Canny/zone')

    # Heatmap saillance par zone
    grid_sal = np.array([
        [sal[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
         for c in range(GRID_N)] for r in range(GRID_N)
    ])
    axes[1,4].imshow(grid_sal, cmap='hot', interpolation='nearest')
    for r in range(GRID_N):
        for c in range(GRID_N):
            axes[1,4].text(c, r, f'{grid_sal[r,c]:.2f}',
                           ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    axes[1,4].set_title('Saillance par zone')

    for ax in axes.flatten(): ax.axis('off')
    plt.suptitle('Analyse spatiale d\'une image', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_segmentation(p_train[0])

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — Extraction features IMAGE spatiales (v3)
# ═══════════════════════════════════════════════════════════════════

def features_per_zone(zone_gray, zone_rgb, zone_hsv):
    """Features compactes pour une zone 64×64."""
    # HOG local
    hog_f = hog(zone_gray, orientations=8,
                pixels_per_cell=(HOG_PIXELS, HOG_PIXELS),
                cells_per_block=(2,2), feature_vector=True)
    # HSV local
    h_h = np.histogram(zone_hsv[:,:,0], bins=N_HSV_BINS, range=(0,180))[0].astype(float)
    s_h = np.histogram(zone_hsv[:,:,1], bins=N_HSV_BINS, range=(0,256))[0].astype(float)
    v_h = np.histogram(zone_hsv[:,:,2], bins=N_HSV_BINS, range=(0,256))[0].astype(float)
    hsv_f = np.concatenate([h_h, s_h, v_h])
    hsv_f /= (hsv_f.sum() + 1e-8)
    # LBP local
    lbp   = local_binary_pattern(zone_gray, LBP_POINTS, LBP_RADIUS, method='uniform')
    lbp_f = np.histogram(lbp, bins=LBP_POINTS+2, range=(0, LBP_POINTS+2))[0].astype(float)
    lbp_f /= (lbp_f.sum() + 1e-8)
    # Stats compactes
    stats = np.array([
        zone_gray.mean(), zone_gray.std(),
        zone_rgb.mean(axis=(0,1)).mean(),  # luminosité moyenne RGB
        zone_hsv[:,:,1].mean(),            # saturation moyenne
    ])
    return np.concatenate([hog_f, hsv_f, lbp_f, stats])


def extract_spatial_features(img_path):
    """
    Features image v3 :
    A) Grille 3×3 : features locales × 9 zones
    B) Segmentation globale : Canny, Deriche, Adaptatif, Otsu, Watershed
    C) Saliency map : distribution spatiale de l'attention
    D) Localisation d'objets : centre de masse, moments spatiaux
    E) Asymétries de composition
    """
    try:
        img = io.imread(img_path)
        if img.ndim == 2:      img = np.stack([img]*3, axis=-1)
        if img.shape[2] == 4:  img = img[:,:,:3]
        img  = (transform.resize(img, IMG_SIZE, anti_aliasing=True) * 255).astype(np.uint8)
        gray = rgb2gray(img)  # float [0,1]
        hsv  = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        h, w = img.shape[:2]
        zh, zw = h // GRID_N, w // GRID_N

        # ════════════════════════════════════════════════════
        # A) GRILLE 3×3 : features locales
        # ════════════════════════════════════════════════════
        zone_feats = []
        for r in range(GRID_N):
            for c in range(GRID_N):
                zg  = gray[r*zh:(r+1)*zh, c*zw:(c+1)*zw]
                zr  = img [r*zh:(r+1)*zh, c*zw:(c+1)*zw]
                zh2 = hsv [r*zh:(r+1)*zh, c*zw:(c+1)*zw]
                zone_feats.append(features_per_zone(zg, zr, zh2))
        zone_vector = np.concatenate(zone_feats)  # 9 × dim_zone

        # ════════════════════════════════════════════════════
        # B) SEGMENTATION MULTI-MÉTHODES (features globales)
        # ════════════════════════════════════════════════════

        # Canny
        canny_map  = segment_canny(gray)
        canny_grid = np.array([
            canny_map[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
            for r in range(GRID_N) for c in range(GRID_N)
        ])  # 9 valeurs : densité de contours par zone

        # Deriche (Scharr)
        deriche_map  = segment_deriche(gray)
        deriche_grid = np.array([
            deriche_map[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
            for r in range(GRID_N) for c in range(GRID_N)
        ])  # 9 valeurs : énergie gradient par zone

        # Adaptatif
        adapt_map  = segment_adaptive(gray)
        adapt_grid = np.array([
            adapt_map[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
            for r in range(GRID_N) for c in range(GRID_N)
        ])  # 9 valeurs : ratio pixels clairs par zone

        # Otsu global
        otsu_map   = segment_otsu(gray)
        otsu_ratio = otsu_map.mean()  # 1 valeur : ratio objet/fond
        otsu_grid  = np.array([
            otsu_map[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
            for r in range(GRID_N) for c in range(GRID_N)
        ])  # 9 valeurs : ratio objet par zone

        # ════════════════════════════════════════════════════
        # C) SALIENCY MAP : distribution spatiale de l'attention
        # ════════════════════════════════════════════════════
        sal_map  = compute_saliency_map(gray)
        sal_grid = np.array([
            sal_map[r*zh:(r+1)*zh, c*zw:(c+1)*zw].mean()
            for r in range(GRID_N) for c in range(GRID_N)
        ])  # 9 valeurs : attention par zone

        # Entropie spatiale de la saillance
        sal_entropy = scipy_entropy(sal_grid + 1e-8)

        # ════════════════════════════════════════════════════
        # D) LOCALISATION D'OBJETS via contours Canny
        # ════════════════════════════════════════════════════
        canny_coords = np.argwhere(canny_map > 0.5)
        if len(canny_coords) > 0:
            cy_norm = canny_coords[:, 0].mean() / h   # centre masse Y normalisé [0,1]
            cx_norm = canny_coords[:, 1].mean() / w   # centre masse X normalisé [0,1]
            cy_std  = canny_coords[:, 0].std()  / h   # dispersion Y
            cx_std  = canny_coords[:, 1].std()  / w   # dispersion X
        else:
            cy_norm = cx_norm = 0.5
            cy_std  = cx_std  = 0.0

        # ════════════════════════════════════════════════════
        # E) ASYMÉTRIES DE COMPOSITION
        # ════════════════════════════════════════════════════
        half_h, half_w = h//2, w//2

        # Asymétrie gauche vs droite
        left_energy   = canny_map[:, :half_w].mean()
        right_energy  = canny_map[:, half_w:].mean()
        asym_lr_canny = left_energy - right_energy

        left_sal  = sal_map[:, :half_w].mean()
        right_sal = sal_map[:, half_w:].mean()
        asym_lr_sal = left_sal - right_sal

        # Asymétrie haut vs bas
        top_energy    = canny_map[:half_h, :].mean()
        bot_energy    = canny_map[half_h:, :].mean()
        asym_tb_canny = top_energy - bot_energy

        top_sal = sal_map[:half_h, :].mean()
        bot_sal = sal_map[half_h:, :].mean()
        asym_tb_sal = top_sal - bot_sal

        # Couleur haut vs bas (utile pour ciel/sol)
        top_hue = hsv[:half_h, :, 0].mean() / 180.0
        bot_hue = hsv[half_h:, :, 0].mean() / 180.0
        top_sat = hsv[:half_h, :, 1].mean() / 255.0
        bot_sat = hsv[half_h:, :, 1].mean() / 255.0
        top_val = hsv[:half_h, :, 2].mean() / 255.0
        bot_val = hsv[half_h:, :, 2].mean() / 255.0
        asym_hue = top_hue - bot_hue
        asym_sat = top_sat - bot_sat
        asym_val = top_val - bot_val

        # ════════════════════════════════════════════════════
        # F) WATERSHED : nombre et taille des régions
        # ════════════════════════════════════════════════════
        try:
            ws_map, _ = segment_watershed(gray)
            n_regions   = ws_map.max()
            regions     = regionprops(ws_map)
            if regions:
                areas = [r.area for r in regions]
                ws_area_mean = np.mean(areas) / (h * w)
                ws_area_std  = np.std(areas)  / (h * w)
                # Position des régions : centre de masse normalisé
                centroids_y = [r.centroid[0] / h for r in regions]
                centroids_x = [r.centroid[1] / w for r in regions]
                ws_cy_mean  = np.mean(centroids_y)
                ws_cx_mean  = np.mean(centroids_x)
                ws_cy_std   = np.std(centroids_y)
                ws_cx_std   = np.std(centroids_x)
            else:
                ws_area_mean = ws_area_std = 0.0
                ws_cy_mean = ws_cx_mean = 0.5
                ws_cy_std = ws_cx_std = 0.0
        except Exception:
            n_regions = 1
            ws_area_mean = ws_area_std = 0.0
            ws_cy_mean = ws_cx_mean = 0.5
            ws_cy_std = ws_cx_std = 0.0

        # ════════════════════════════════════════════════════
        # G) FEATURES GLOBALES complémentaires
        # ════════════════════════════════════════════════════
        edge_density_global = canny_map.mean()
        texture_entropy     = scipy_entropy(
            np.histogram(gray.flatten(), bins=32)[0].astype(float) + 1e-8
        )
        brightness_mean = gray.mean()
        brightness_std  = gray.std()

        # HOG global (v1)
        hog_global = hog(gray, orientations=9,
                         pixels_per_cell=(16,16), cells_per_block=(2,2),
                         feature_vector=True)

        # ════════════════════════════════════════════════════
        # Assembler tout
        # ════════════════════════════════════════════════════
        spatial_scalars = np.array([
            # Localisation contours
            cy_norm, cx_norm, cy_std, cx_std,
            # Asymétries
            asym_lr_canny, asym_tb_canny,
            asym_lr_sal,   asym_tb_sal,
            asym_hue, asym_sat, asym_val,
            top_hue, bot_hue, top_sat, bot_sat, top_val, bot_val,
            # Watershed
            n_regions / 50.0,  # normalisé
            ws_area_mean, ws_area_std,
            ws_cy_mean, ws_cx_mean, ws_cy_std, ws_cx_std,
            # Saliency
            sal_entropy,
            sal_map.mean(), sal_map.std(),
            sal_map.max(),
            # Global
            edge_density_global, otsu_ratio,
            texture_entropy, brightness_mean, brightness_std,
        ])

        return np.concatenate([
            zone_vector,       # A : grille 3×3
            canny_grid,        # B1 : Canny par zone (9)
            deriche_grid,      # B2 : Deriche par zone (9)
            adapt_grid,        # B3 : Adaptatif par zone (9)
            otsu_grid,         # B4 : Otsu par zone (9)
            sal_grid,          # C : Saillance par zone (9)
            spatial_scalars,   # D+E+F+G
            hog_global,        # compatibilité v1
        ])

    except Exception as e:
        print(f"Erreur image {img_path}: {e}")
        return np.zeros(500)  # dimension approx, sera géré par VT


# Test
_s = extract_spatial_features(p_train[0])
print(f"Shape features image v3 : {_s.shape}")

print("Extraction features images (train)...")
F_img_train = np.array([extract_spatial_features(p) for p in p_train])
print("Extraction features images (val)...")
F_img_val   = np.array([extract_spatial_features(p) for p in p_val])
print("Extraction features images (test)...")
F_img_test  = np.array([extract_spatial_features(p) for p in p_test])

print(f"Shape finale features image : {F_img_train.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 — Extraction features TEXTE avec mots spatiaux (v3)
# ═══════════════════════════════════════════════════════════════════

# Vocabulaires spatiaux et visuels
SPATIAL_WORDS = {
    'left':    0, 'right':   1, 'top':     2, 'bottom':  3,
    'upper':   2, 'lower':   3, 'above':   2, 'below':   3,
    'center':  4, 'middle':  4, 'front':   5, 'behind':  6,
    'back':    6, 'foreground': 5, 'background': 6,
    'corner':  7, 'edge':    8, 'side':    8,
    'near':    9, 'far':    10, 'close':   9, 'distant': 10,
}

COLOR_WORDS = {
    'red':1,'blue':1,'green':1,'yellow':1,'orange':1,'purple':1,
    'white':1,'black':1,'gray':1,'grey':1,'pink':1,'brown':1,
    'sky':1,'dark':1,'bright':1,'light':1,'colorful':1,'pale':1,
    'golden':1,'silver':1,'vivid':1,'dull':1,
}

SCENE_WORDS = {
    'sky':    'top',    'cloud':  'top',    'sun':    'top',
    'moon':   'top',    'ceiling':'top',    'floor':  'bottom',
    'ground': 'bottom', 'road':   'bottom', 'water':  'bottom',
    'grass':  'bottom', 'tree':   'center', 'building':'center',
    'person': 'center', 'people': 'center', 'man':    'center',
    'woman':  'center', 'child':  'center', 'animal': 'center',
    'dog':    'bottom', 'cat':    'bottom', 'bird':   'top',
    'mountain':'center','river':  'bottom', 'ocean':  'bottom',
    'horizon': 'center',
}


def extract_text_spatial(text):
    """Features texte enrichies avec conscience spatiale."""
    tokens       = word_tokenize(text.lower())
    alpha_tokens = [t for t in tokens if t.isalpha()]
    sentences    = sent_tokenize(text)
    tagged       = pos_tag(alpha_tokens) if alpha_tokens else []

    nouns  = [w for w,t in tagged if t.startswith('NN')]
    verbs  = [w for w,t in tagged if t.startswith('VB')]
    adjs   = [w for w,t in tagged if t.startswith('JJ')]
    advs   = [w for w,t in tagged if t.startswith('RB')]
    n = len(alpha_tokens) + 1

    # ── v1 features ──────────────────────────────────────────────
    unique_ratio  = len(set(alpha_tokens)) / n
    avg_word_len  = np.mean([len(w) for w in alpha_tokens]) if alpha_tokens else 0
    stop_tokens   = [t for t in tokens if t in STOP]
    punct_count   = sum(1 for t in tokens if not t.isalnum())

    # ── Sentiment ────────────────────────────────────────────────
    if VADER_OK:
        vs = vader.polarity_scores(text)
        s_pos, s_neg, s_neu, s_comp = vs['pos'], vs['neg'], vs['neu'], vs['compound']
    else:
        s_pos = s_neg = s_neu = s_comp = 0.0

    # ── Mots spatiaux (v3) ───────────────────────────────────────
    spatial_counts = np.zeros(11)  # 11 catégories spatiales
    for tok in alpha_tokens:
        if tok in SPATIAL_WORDS:
            spatial_counts[SPATIAL_WORDS[tok]] += 1
    spatial_counts /= (n + 1)

    has_left    = int(any(t in ['left','leftward','leftmost'] for t in alpha_tokens))
    has_right   = int(any(t in ['right','rightward','rightmost'] for t in alpha_tokens))
    has_top     = int(any(t in ['top','upper','above','sky','overhead'] for t in alpha_tokens))
    has_bottom  = int(any(t in ['bottom','lower','below','ground','floor'] for t in alpha_tokens))
    has_center  = int(any(t in ['center','middle','central'] for t in alpha_tokens))
    n_spatial   = sum([has_left, has_right, has_top, has_bottom, has_center])

    # ── Mots de couleur ──────────────────────────────────────────
    color_count = sum(1 for t in alpha_tokens if t in COLOR_WORDS) / n

    # ── Scène et position attendue des objets ────────────────────
    expected_positions = {'top':0, 'bottom':0, 'center':0}
    for tok in alpha_tokens:
        if tok in SCENE_WORDS:
            pos = SCENE_WORDS[tok]
            if pos in expected_positions:
                expected_positions[pos] += 1
    tot_scene = sum(expected_positions.values()) + 1
    exp_top    = expected_positions['top']    / tot_scene
    exp_bottom = expected_positions['bottom'] / tot_scene
    exp_center = expected_positions['center'] / tot_scene
    scene_diversity = scipy_entropy(np.array(list(expected_positions.values())) + 1e-8)

    # ── Readability + structure ───────────────────────────────────
    n_sent = max(len(sentences), 1)
    n_syl  = sum(max(1, len([c for c in w if c in 'aeiouAEIOU'])) for w in alpha_tokens)
    flesch = 206.835 - 1.015*(len(alpha_tokens)/n_sent) - 84.6*(n_syl/max(n,1))
    avg_sent_len = len(alpha_tokens) / n_sent

    v1 = np.array([
        len(alpha_tokens), len(text.strip()),
        len(nouns)/n, len(verbs)/n, len(adjs)/n, len(advs)/n,
        len(nouns)/(len(verbs)+1), unique_ratio, avg_word_len,
        len(stop_tokens)/n, punct_count/n,
        text.count('?'), text.count('!'),
        int(text.strip().endswith('.')),
        s_pos, s_neg, s_neu, s_comp,
        flesch, avg_sent_len,
        color_count,
    ])

    v3_spatial = np.array([
        has_left, has_right, has_top, has_bottom, has_center,
        n_spatial,
        exp_top, exp_bottom, exp_center,
        scene_diversity,
    ])

    return np.concatenate([v1, spatial_counts, v3_spatial])


# TF-IDF
print("Extraction TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=2000, ngram_range=(1,2),
    stop_words='english', sublinear_tf=True, min_df=3
)
T_tfidf_train = tfidf.fit_transform(t_train).toarray()
T_tfidf_val   = tfidf.transform(t_val).toarray()
T_tfidf_test  = tfidf.transform(t_test).toarray()

print("Extraction stats texte spatiales...")
S_train = np.array([extract_text_spatial(t) for t in t_train])
S_val   = np.array([extract_text_spatial(t) for t in t_val])
S_test  = np.array([extract_text_spatial(t) for t in t_test])

F_text_train = np.hstack([T_tfidf_train, S_train])
F_text_val   = np.hstack([T_tfidf_val,   S_val])
F_text_test  = np.hstack([T_tfidf_test,  S_test])

print(f"Shape features texte : {F_text_train.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6 — Cross-features spatiaux texte ↔ image (CLEF v3)
# ═══════════════════════════════════════════════════════════════════
#
# Ces features COMPARENT directement ce que dit le texte
# avec ce qu'on voit dans l'image → signal de cohérence direct

# Index des features spatiales dans S_train
# (basé sur extract_text_spatial)
IDX_V1    = 21
IDX_SPAT  = IDX_V1 + 11   # spatial_counts (11)
IDX_SCENE = IDX_SPAT       # = IDX_V1+11
# has_left=IDX_V1+0, has_right=IDX_V1+1, has_top=IDX_V1+2,
# has_bottom=IDX_V1+3, has_center=IDX_V1+4
# spatial_counts[0..10] commence à IDX_V1+6
IDX_HAS   = IDX_V1  # has_left, has_right, has_top, has_bottom, has_center
IDX_EXP   = IDX_SPAT + 6  # exp_top, exp_bottom, exp_center

def build_cross_spatial_features(F_img, S_text, V_text_cca, V_img_cca):
    """
    Features de cross-cohérence spatiale :
    1. Confronte positions mentionnées dans le texte vs zones actives dans l'image
    2. Compare les asymétries texte/image
    3. Distances dans l'espace CCA (comme v2)
    4. Distances scalaires enrichies
    """
    N = len(F_img)
    features = []

    # Indices dans F_img pour les cartes de zones (après zone_vector)
    # zone_vector = 9 × dim_zone ; canny_grid commence à zone_vector.size
    # On accède via S_text (les stats extraites)

    for i in range(N):
        t  = V_text_cca[i]  # vecteur CCA texte
        v  = V_img_cca[i]   # vecteur CCA image
        d  = t - v
        st = S_text[i]      # stats texte brutes
        fi = F_img[i]       # features image brutes

        # ── Distances CCA (comme v2) ──────────────────────────────
        dist_cosine      = float(cosine(t, v))
        cos_sim          = 1 - dist_cosine
        dist_euclidean   = np.linalg.norm(d)
        dist_manhattan   = float(cityblock(t, v))
        dist_chebyshev   = float(chebyshev(t, v))
        dist_canberra    = float(canberra(t, v))
        dist_braycurtis  = float(braycurtis(t, v))
        dot_product      = np.dot(t, v)
        angle            = np.arccos(np.clip(cos_sim, -1, 1))
        pearson_r        = pearsonr(t, v)[0] if len(t) > 2 else 0.0
        norm_ratio       = np.linalg.norm(t) / (np.linalg.norm(v) + 1e-8)

        # Projection orthogonale
        proj_scalar  = np.dot(t, v) / (np.linalg.norm(v) + 1e-8)
        proj_vec     = proj_scalar * v / (np.linalg.norm(v) + 1e-8)
        orth_norm    = np.linalg.norm(t - proj_vec)

        diff_std = d.std()
        product  = t * v
        abs_diff = np.abs(d)

        # ── Cross-features spatiaux (NOUVEAUTÉ v3) ────────────────
        # Récupérer les maps de zones depuis F_img
        # Les 45 dernières valeurs avant hog_global = 5 cartes × 9 zones
        # Canny_grid = F_img[zone_size : zone_size+9]
        # On calcule zone_size dynamiquement :
        n_cca_feats = len(t) * 3   # d, product, abs_diff
        # Feature brutes : cartes de zones (positions dans F_img)
        # Canny est à zone_vector.size, mais on ne connait pas zone_size
        # → utiliser les asymétries (spatial_scalars) dans F_img
        # On a placé spatial_scalars en avant-dernière position
        # On utilise plutôt les stats texte spatiales vs asymétries image

        # Asymétries image (extraites des spatial_scalars)
        # spatial_scalars commence à : zone_vector + 5×9 = zone_vector + 45
        # On ne peut pas facilement indexer sans stocker les dims
        # → on utilise les stats brutes S_text comme proxy texte
        # et les features image en fin de vecteur

        # Proxy : le texte mentionne-t-il le haut ? (sky, cloud, sun)
        # Compare avec l'asymétrie haut/bas de la saillance dans l'image
        # Ces deux indices sont dans S_text
        text_exp_top    = float(st[IDX_EXP + 0]) if len(st) > IDX_EXP+2 else 0.0
        text_exp_bottom = float(st[IDX_EXP + 1]) if len(st) > IDX_EXP+2 else 0.0
        text_exp_center = float(st[IDX_EXP + 2]) if len(st) > IDX_EXP+2 else 0.0
        text_has_left   = float(st[IDX_HAS + 0]) if len(st) > IDX_HAS+4 else 0.0
        text_has_right  = float(st[IDX_HAS + 1]) if len(st) > IDX_HAS+4 else 0.0

        # Asymétries image (les 9 derniers scalars spatiaux)
        # asym_lr_canny, asym_tb_canny, asym_lr_sal, asym_tb_sal
        # asym_hue, asym_sat, asym_val, top_hue, bot_hue...
        # Leurs positions dans F_img : fin du vecteur - hog_global.size - len(spatial_scalars)
        # Pour simplifier, on stocke les asymétries séparément
        # → on les recalcule depuis F_img en fin de vecteur
        # Index : F_img[-hog_len - n_spatial_scalars : -hog_len]
        # n_spatial_scalars = 33 (compter dans extract_spatial_features)
        # Indices des asymétries dans spatial_scalars :
        # 0:cy_norm,1:cx_norm,2:cy_std,3:cx_std
        # 4:asym_lr_canny, 5:asym_tb_canny, 6:asym_lr_sal, 7:asym_tb_sal
        # 8:asym_hue,9:asym_sat,10:asym_val
        # 11:top_hue,12:bot_hue,13:top_sat,14:bot_sat,15:top_val,16:bot_val
        # 17...: watershed, saliency, global
        N_HOG  = 81   # hog(192×192, 9 orient, 16 ppc, 2 cpb) = ((192/16-1)²*4 * 9
        N_SPAT = 33
        spat_start = -(N_HOG + N_SPAT)
        spat_end   = -N_HOG if N_HOG > 0 else None
        if len(fi) > N_HOG + N_SPAT:
            spat = fi[spat_start:spat_end]
            cy_norm      = spat[0] if len(spat) > 0 else 0.5
            cx_norm      = spat[1] if len(spat) > 1 else 0.5
            asym_lr_canny = spat[4] if len(spat) > 4 else 0.0
            asym_tb_canny = spat[5] if len(spat) > 5 else 0.0
            asym_tb_sal   = spat[7] if len(spat) > 7 else 0.0
            asym_hue      = spat[8] if len(spat) > 8 else 0.0
            top_hue       = spat[11] if len(spat) > 11 else 0.0
            bot_hue       = spat[12] if len(spat) > 12 else 0.0
        else:
            cy_norm = cx_norm = 0.5
            asym_lr_canny = asym_tb_canny = 0.0
            asym_tb_sal = asym_hue = 0.0
            top_hue = bot_hue = 0.0

        # Cross-features :
        # Si texte dit "sky" (exp_top élevé) → image devrait avoir
        # les contours/activité EN HAUT (asym_tb < 0 = plus actif en haut)
        cross_top_coherence    = text_exp_top * (-asym_tb_canny)   # cohérent si les deux sont +
        cross_bottom_coherence = text_exp_bottom * asym_tb_canny
        cross_left_right       = (text_has_left - text_has_right) * asym_lr_canny
        # Couleur ciel : si texte mentionne ciel/bleu → top_hue devrait être bleu (hue~120/180)
        has_sky_word = int(any(w in ['sky','cloud','blue','sun','sunset','sunrise']
                               for w in word_tokenize(text.lower())))
        sky_hue_match = has_sky_word * (1 - abs(top_hue - 0.56))  # 0.56 = bleu/cyan normalisé

        # Position centre de masse des contours vs attente texte
        # Si texte attend objet en bas → cy_norm devrait être > 0.5
        pos_coherence = (text_exp_bottom - text_exp_top) * (cy_norm - 0.5)

        cross_feats = np.array([
            cross_top_coherence, cross_bottom_coherence,
            cross_left_right, sky_hue_match,
            pos_coherence,
            text_exp_top * top_hue,      # ciel attendu × teinte haut réelle
            text_exp_bottom * bot_hue,   # sol attendu × teinte bas réelle
            cx_norm, cy_norm,            # position brute du centre de masse
        ])

        # ── Vecteurs diff/product/abs_diff (CCA) ─────────────────
        scalars = np.array([
            dist_cosine, cos_sim, dist_euclidean,
            dist_manhattan, dist_chebyshev,
            dist_canberra, dist_braycurtis,
            dot_product, angle, pearson_r,
            norm_ratio, proj_scalar, orth_norm,
            diff_std, d.mean(), d.max(), d.min(),
        ])

        features.append(np.concatenate([
            scalars, cross_feats,
            d, product, abs_diff
        ]))

    return np.array(features)


print("Cross-features définis — prêt pour la projection CCA.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7 — Nettoyage features + PCA + CCA
# ═══════════════════════════════════════════════════════════════════

def remove_correlated_features(X, threshold=0.95):
    X_s = X[:min(500, len(X))]
    corr = np.corrcoef(X_s.T)
    np.fill_diagonal(corr, 0)
    to_drop = set()
    for i in range(corr.shape[0]):
        if i in to_drop: continue
        for j in range(i+1, corr.shape[1]):
            if abs(corr[i,j]) > threshold: to_drop.add(j)
    return [i for i in range(X.shape[1]) if i not in to_drop]


print("Nettoyage des features...")

# VarianceThreshold
vt_img  = VarianceThreshold(threshold=1e-6)
vt_text = VarianceThreshold(threshold=1e-6)
F_img_train_vt  = vt_img.fit_transform(F_img_train)
F_img_val_vt    = vt_img.transform(F_img_val)
F_img_test_vt   = vt_img.transform(F_img_test)
F_text_train_vt = vt_text.fit_transform(F_text_train)
F_text_val_vt   = vt_text.transform(F_text_val)
F_text_test_vt  = vt_text.transform(F_text_test)
print(f"Image  : {F_img_train.shape[1]} → {F_img_train_vt.shape[1]}")
print(f"Texte  : {F_text_train.shape[1]} → {F_text_train_vt.shape[1]}")

# Normaliser
sc_img  = StandardScaler()
sc_text = StandardScaler()
F_img_train_s  = sc_img.fit_transform(F_img_train_vt)
F_img_val_s    = sc_img.transform(F_img_val_vt)
F_img_test_s   = sc_img.transform(F_img_test_vt)
F_text_train_s = sc_text.fit_transform(F_text_train_vt)
F_text_val_s   = sc_text.transform(F_text_val_vt)
F_text_test_s  = sc_text.transform(F_text_test_vt)

# Dé-duplication (image seulement — TF-IDF épargné)
keep_img = remove_correlated_features(F_img_train_s, threshold=0.95)
F_img_train_s = F_img_train_s[:, keep_img]
F_img_val_s   = F_img_val_s[:,   keep_img]
F_img_test_s  = F_img_test_s[:,  keep_img]
print(f"Image après dédup : {len(keep_img)} features")

# PCA pré-CCA
K_pre = 128
K_cca = 64
print(f"\nPCA pré-CCA : {K_pre} dims...")
pca_img_pre  = PCA(n_components=min(K_pre, F_img_train_s.shape[1]),  random_state=42)
pca_text_pre = PCA(n_components=min(K_pre, F_text_train_s.shape[1]), random_state=42)
P_img_train  = pca_img_pre.fit_transform(F_img_train_s)
P_img_val    = pca_img_pre.transform(F_img_val_s)
P_img_test   = pca_img_pre.transform(F_img_test_s)
P_text_train = pca_text_pre.fit_transform(F_text_train_s)
P_text_val   = pca_text_pre.transform(F_text_val_s)
P_text_test  = pca_text_pre.transform(F_text_test_s)

print(f"CCA : {K_cca} composantes canoniques...")
cca = CCA(n_components=K_cca, max_iter=1000)
cca.fit(P_text_train, P_img_train)
V_text_train, V_img_train = cca.transform(P_text_train, P_img_train)
V_text_val,   V_img_val   = cca.transform(P_text_val,   P_img_val)
V_text_test,  V_img_test  = cca.transform(P_text_test,  P_img_test)
V_img_train  = normalize(V_img_train);  V_img_val  = normalize(V_img_val)
V_img_test   = normalize(V_img_test);   V_text_train = normalize(V_text_train)
V_text_val   = normalize(V_text_val);   V_text_test  = normalize(V_text_test)
print(f"Shape vecteurs CCA : {V_text_train.shape}")

# Top-5 corrélations canoniques
print("Top-5 corrélations canoniques :")
for i in range(min(5, K_cca)):
    r = np.corrcoef(V_text_train[:,i], V_img_train[:,i])[0,1]
    print(f"  Composante {i+1} : r = {r:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 8 — Construction du dataset final
# ═══════════════════════════════════════════════════════════════════
print("Construction dataset final (train)...")
D_train = build_cross_spatial_features(F_img_train, S_train, V_text_train, V_img_train)
print("Construction dataset final (val)...")
D_val   = build_cross_spatial_features(F_img_val,   S_val,   V_text_val,   V_img_val)
print("Construction dataset final (test)...")
D_test  = build_cross_spatial_features(F_img_test,  S_test,  V_text_test,  V_img_test)
print(f"Shape dataset : {D_train.shape}")

# Normaliser
sc_dist = StandardScaler()
X_train = sc_dist.fit_transform(D_train)
X_val   = sc_dist.transform(D_val)
X_test  = sc_dist.transform(D_test)

# VarianceThreshold + SelectFromModel
vt_final = VarianceThreshold(threshold=1e-4)
X_train  = vt_final.fit_transform(X_train)
X_val    = vt_final.transform(X_val)
X_test   = vt_final.transform(X_test)
print(f"Après VT : {X_train.shape[1]} features")

selector = SelectFromModel(
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    threshold='mean'
)
selector.fit(X_train, y_train)
X_train_sel = selector.transform(X_train)
X_val_sel   = selector.transform(X_val)
X_test_sel  = selector.transform(X_test)
print(f"Après sélection RF : {X_train_sel.shape[1]} features conservées")
print(f"Réduction : {(1 - X_train_sel.shape[1]/X_train.shape[1])*100:.1f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 9 — Entraînement et comparaison
# ═══════════════════════════════════════════════════════════════════
import copy

MODELS = {
    'Logistic Regression':  LogisticRegression(max_iter=2000, C=1.0, random_state=42),
    'Linear SVM':           LinearSVC(max_iter=5000, C=1.0, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=300, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=200,
                                                        max_depth=4, random_state=42),
}

def eval_models(X_tr, X_vl, label=''):
    print(f"\n── {label} ──")
    print(f"{'Modèle':<25} {'CV3':>8} {'±':>5} {'ValAcc':>8} {'ValF1':>8}")
    print("-" * 60)
    results = {}
    for name, mdl in MODELS.items():
        m = copy.deepcopy(mdl)
        cv = cross_val_score(m, X_tr, y_train, cv=3, scoring='accuracy', n_jobs=1)
        m.fit(X_tr, y_train)
        yp  = m.predict(X_vl)
        acc = accuracy_score(y_val, yp)
        f1  = f1_score(y_val, yp)
        print(f"{name:<25} {cv.mean():.4f} {cv.std():.3f} {acc:.4f} {f1:.4f}")
        results[name] = {'cv':cv.mean(), 'acc':acc, 'f1':f1, 'model':m}
    return results

res_full = eval_models(X_train,     X_val,     'Dataset complet')
res_sel  = eval_models(X_train_sel, X_val_sel, 'Dataset sélectionné')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 10 — Évaluation finale TEST
# ═══════════════════════════════════════════════════════════════════
all_res = {**res_full, **{n+'[sel]': v for n,v in res_sel.items()}}
best_name  = max(all_res, key=lambda k: all_res[k]['acc'])
best_model = all_res[best_name]['model']
use_sel    = '[sel]' in best_name
X_test_final = X_test_sel if use_sel else X_test

y_pred   = best_model.predict(X_test_final)
test_acc = accuracy_score(y_test, y_pred)
test_f1  = f1_score(y_test, y_pred)

print("═" * 60)
print(f"  RÉSULTAT FINAL — Pipeline v3 Spatial + CCA")
print(f"  Modèle      : {best_name}")
print(f"  Test Acc    : {test_acc:.4f}")
print(f"  Test F1     : {test_f1:.4f}")
print("═" * 60)
print(classification_report(y_test, y_pred,
      target_names=['incohérent','cohérent']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['incohérent','cohérent'],
            yticklabels=['incohérent','cohérent'])
plt.title(f'Confusion — {best_name}')
plt.ylabel('Réel'); plt.xlabel('Prédit')
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 11 — Importance des features + analyse spatiale
# ═══════════════════════════════════════════════════════════════════
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    imp = np.abs(best_model.coef_[0])
else:
    imp = None

if imp is not None:
    top_k = min(30, len(imp))
    top_idx = np.argsort(imp)[::-1][:top_k]

    plt.figure(figsize=(18, 5))
    plt.bar(range(top_k), imp[top_idx], color='steelblue')
    plt.xticks(range(top_k), [f'f{i}' for i in top_idx],
               rotation=45, ha='right', fontsize=8)
    plt.title('Top-30 features les plus importantes')
    plt.tight_layout()
    plt.show()

    print("Top-15 features (index dans le vecteur final) :")
    scalar_names = [
        'dist_cosine','cos_sim','dist_euclidean','dist_manhattan','dist_chebyshev',
        'dist_canberra','dist_braycurtis','dot_product','angle','pearson_r',
        'norm_ratio','proj_scalar','orth_norm','diff_std','diff_mean','diff_max','diff_min',
        'cross_top','cross_bottom','cross_left_right','sky_hue_match','pos_coherence',
        'txt_exp_top*top_hue','txt_exp_bot*bot_hue','cx_norm','cy_norm',
    ]
    for rank, idx in enumerate(top_idx[:15]):
        name = scalar_names[idx] if idx < len(scalar_names) else f'cca_dim_{idx-len(scalar_names)}'
        print(f"  {rank+1:2d}. [{idx:4d}] {name:<30} imp={imp[idx]:.6f}")

    # Cross-features spatiaux : sont-ils utiles ?
    cross_start = 17  # index de cross_top dans scalars
    cross_end   = 26
    if len(imp) > cross_end:
        cross_imp = imp[cross_start:cross_end].mean()
        total_imp = imp.mean()
        print(f"\nImportance moyenne cross-features spatiaux : {cross_imp:.6f}")
        print(f"Importance moyenne totale                  : {total_imp:.6f}")
        print(f"Ratio spatial/total                        : {cross_imp/total_imp:.2f}x")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 12 — Visualisation : cas bien classés vs mal classés
# ═══════════════════════════════════════════════════════════════════
y_pred_val = best_model.predict(X_val_sel if use_sel else X_val)

correct   = np.where(y_pred_val == y_val)[0]
incorrect = np.where(y_pred_val != y_val)[0]

print(f"Val : {len(correct)} corrects / {len(incorrect)} erreurs")

def show_examples(indices, title, n=4):
    idx = np.random.choice(indices, min(n, len(indices)), replace=False)
    fig, axes = plt.subplots(2, len(idx), figsize=(4*len(idx), 8))
    for k, i in enumerate(idx):
        img = io.imread(p_val[i])
        if img.ndim == 2: img = np.stack([img]*3, axis=-1)
        if img.shape[2] == 4: img = img[:,:,:3]
        axes[0,k].imshow(img)
        axes[0,k].set_title(f"Réel:{['Inc','Coh'][y_val[i]]} / Prédit:{['Inc','Coh'][y_pred_val[i]]}",
                            fontsize=9, color='green' if y_pred_val[i]==y_val[i] else 'red')
        axes[0,k].axis('off')
        txt = t_val[i][:120] + '...' if len(t_val[i]) > 120 else t_val[i]
        axes[1,k].text(0.5, 0.5, txt, ha='center', va='center',
                       wrap=True, fontsize=8)
        axes[1,k].axis('off')
    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

show_examples(correct,   'Exemples bien classés',  n=4)
show_examples(incorrect, 'Exemples mal classés', n=4)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 13 — Sauvegarde
# ═══════════════════════════════════════════════════════════════════
import joblib
joblib.dump(best_model,    'v3_best_model.pkl')
joblib.dump(sc_dist,       'v3_scaler_dist.pkl')
joblib.dump(sc_img,        'v3_scaler_img.pkl')
joblib.dump(sc_text,       'v3_scaler_text.pkl')
joblib.dump(pca_img_pre,   'v3_pca_img.pkl')
joblib.dump(pca_text_pre,  'v3_pca_text.pkl')
joblib.dump(cca,           'v3_cca.pkl')
joblib.dump(tfidf,         'v3_tfidf.pkl')
joblib.dump(vt_img,        'v3_vt_img.pkl')
joblib.dump(vt_text,       'v3_vt_text.pkl')
joblib.dump(selector,      'v3_selector.pkl')

print("Sauvegarde OK.")
print(f"\nTest Accuracy : {test_acc:.4f}")
print(f"Test F1       : {test_f1:.4f}")
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("Pipeline v3 complet :")
print("  IMAGE  : Grille 3×3 (HOG+HSV+LBP local)")
print("           + Canny/Deriche/Adaptatif/Otsu/Watershed/Saliency")
print("           + Localisation objets (centre masse, asymétries)")
print("  TEXTE  : TF-IDF + Stats + Mots spatiaux/couleurs/scène")
print("  CROSS  : Cohérence spatiale texte↔image directe")
print("  PROJ   : PCA(128) → CCA(64) conjoint")
print("  MODÈLE : classifieur sur distances + cross-features")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")